# PyTorch ANN Template — Reusable for Classification or Regression

## What this notebook is

A single, self-contained template for training a feed-forward neural network
(ANN/MLP) on tabular data with PyTorch, structured the way a production ML
codebase would be organized — just flattened into notebook cells so every
piece is visible and explained.

## How to reuse it for a new project

1. Set `config.task_type` to `"classification"` or `"regression"`.
2. Set `config.data_path` to your CSV. Leave it as `None` to run the notebook
   immediately on an auto-generated synthetic dataset — useful for smoke-testing
   the pipeline before you have real data, or right now to see it run end to end.
3. If your target column isn't already named `target`, set `config.target_column`.
4. Run **Kernel → Restart & Run All**. Everything downstream — architecture,
   loss, optimizer, metrics, HPO search space — adapts automatically based on
   `task_type`, so you shouldn't need to touch code below the config cell.

## Design principles followed throughout

- **One responsibility per cell.** Each cell does one thing so you can re-run
  or replace it in isolation.
- **Functions, not copy-paste.** Training/eval/metrics logic is written once
  (`fit_model`, `evaluate`, `compute_metrics`, ...) and reused for the baseline
  run, every Optuna trial, and the final retrain — instead of three
  near-identical training loops that drift out of sync over time.
- **No leakage.** Scaling is fit on the training split only; the test set is
  touched exactly once, for the final numbers.
- **Config-driven, not hardcoded.** Paths, hyperparameters, and task type all
  live in one `Config` object.


## 2. Import Libraries

**Why grouped this way:** standard library → third-party data/ML tooling →
PyTorch → Optuna. This ordering (and keeping each group alphabetized-ish) is
the convention most Python linters (`isort`, `ruff`) enforce, and it makes it
immediately obvious which dependencies a notebook needs.

**Common mistake:** scattering `import` statements throughout the notebook as
you "discover" you need them. It makes dependencies impossible to audit at a
glance and causes `NameError`s if cells are run out of order.


In [ ]:
# --- Standard library ---
import os
import copy
import random
import warnings
from dataclasses import dataclass, field
from typing import Optional, List, Tuple, Dict, Any

# --- Data handling ---
import numpy as np
import pandas as pd

# --- Visualization ---
import matplotlib.pyplot as plt

# --- ML utilities ---
from sklearn.datasets import make_classification, make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    confusion_matrix, ConfusionMatrixDisplay, classification_report,
    mean_squared_error, mean_absolute_error, r2_score,
)

# --- Deep learning ---
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

# --- Hyperparameter optimization ---
import optuna
from optuna.trial import TrialState

warnings.filterwarnings("ignore", category=UserWarning)
print(f"PyTorch {torch.__version__} | Optuna {optuna.__version__}")

## 3. Configuration

**Why a dataclass instead of loose variables or a plain dict:** every
hyperparameter and path lives in one typed, autocompletable object that gets
passed around explicitly. Loose global variables (`LR = 0.001` scattered
across cells) are the single biggest source of "wait, which value is this
actually using?" bugs when a notebook gets re-run out of order.

**Common mistake:** hardcoding a dataset path like `/content/data.csv` deep
inside a loading function. Nothing below this cell should need editing when
you switch datasets or tasks — that's the point of centralizing it here.

**Best practice:** mutable defaults (like a list) need `field(default_factory=...)`
in a dataclass — using `hidden_dims: List[int] = [128, 64]` directly would
share ONE list across every `Config` instance you create, a classic Python
gotcha.


In [ ]:
@dataclass
class Config:
    # --- Task ---
    task_type: str = "classification"        # "classification" or "regression"

    # --- Reproducibility ---
    seed: int = 42

    # --- Data ---
    data_path: Optional[str] = None           # your CSV path; None -> synthetic demo data
    target_column: Optional[str] = None       # set if your target column isn't already "target"
    test_size: float = 0.15
    val_size: float = 0.15                    # both fractions are of the FULL dataset

    # --- DataLoader ---
    batch_size: int = 64
    num_workers: int = 2

    # --- Model (defaults; Optuna searches around these during tuning) ---
    hidden_dims: List[int] = field(default_factory=lambda: [128, 64])
    dropout_rate: float = 0.2
    use_batch_norm: bool = True

    # --- Training ---
    max_epochs: int = 50
    learning_rate: float = 1e-3
    weight_decay: float = 1e-4
    optimizer_name: str = "adamw"              # adamw | adam | sgd | rmsprop
    scheduler_name: str = "cosine"              # none | step | cosine | plateau | onecycle
    early_stopping_patience: int = 7
    use_amp: bool = False                       # mixed precision; only kicks in on CUDA

    # --- Hyperparameter tuning ---
    n_optuna_trials: int = 15
    optuna_timeout_sec: Optional[int] = None

    # --- Output ---
    output_dir: str = "outputs"
    checkpoint_name: str = "best_model.pt"


config = Config()
os.makedirs(config.output_dir, exist_ok=True)
config

## 4. Reproducibility

**Why:** Python's `random`, NumPy, and PyTorch each maintain independent RNG
state. Seeding only `torch.manual_seed` (a very common half-measure) still
leaves `train_test_split`'s shuffling and any NumPy-based augmentation
non-deterministic.

**Common mistake:** seeding once at the top and assuming it's permanent.
Creating new DataLoaders with `shuffle=True`, or any library call that draws
randomness, consumes RNG state — if you need a specific cell to be
reproducible in isolation, reseed right before it.

**Best practice / tradeoff:** `cudnn.deterministic = True` sacrifices some GPU
throughput for bit-for-bit reproducibility. Flip it to `False` (and
`benchmark = True`) once you've finished debugging and just want speed.


In [ ]:
def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


seed_everything(config.seed)

## 5. Device Configuration

**Why:** code that hardcodes `.cuda()` crashes on a CPU-only machine, and code
that hardcodes `.cpu()` wastes a GPU when one's available. Checking once and
reusing a single `device` object everywhere is the standard device-agnostic
pattern.

**Common mistake:** moving the model to `device` but forgetting to move a
batch's tensors too (or vice versa) — this throws a
`RuntimeError: Expected all tensors to be on the same device`.

**Best practice (2026):** also check for Apple Silicon's `mps` backend — a
meaningful fraction of local development now happens on M-series Macs.


In [ ]:
def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")


device = get_device()
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

## 6. Data Loading

**Why:** the notebook needs to be runnable via **Restart & Run All** with zero
setup, but it also needs to be trivially pointed at real data. The solution:
`load_data` reads `config.data_path`; when it's `None`, it generates a
synthetic dataset (matched to `task_type`) so the whole pipeline is exercised
immediately with no external file required.

**Common mistake:** silently swallowing a bad path with something like
`pd.read_csv(path, on_bad_lines="skip")` and continuing — you want a loud,
early failure if the configured path is wrong, not a confusing shape mismatch
five cells later.


In [ ]:
def load_data(config: Config) -> pd.DataFrame:
    """Return a DataFrame with all feature columns plus a 'target' column."""
    if config.data_path is not None:
        if not os.path.exists(config.data_path):
            raise FileNotFoundError(
                f"config.data_path='{config.data_path}' does not exist. "
                "Point it at your CSV, or set it back to None to use synthetic demo data."
            )
        loaded_df = pd.read_csv(config.data_path)
        if config.target_column:
            loaded_df = loaded_df.rename(columns={config.target_column: "target"})
        return loaded_df

    print("config.data_path is None -> generating a synthetic demo dataset "
          "so the notebook runs end to end. Set config.data_path to use real data.")
    if config.task_type == "classification":
        X_demo, y_demo = make_classification(
            n_samples=4000, n_features=20, n_informative=12, n_redundant=4,
            n_classes=4, class_sep=1.2, random_state=config.seed,
        )
    else:
        X_demo, y_demo = make_regression(
            n_samples=4000, n_features=20, n_informative=12, noise=15.0,
            random_state=config.seed,
        )

    loaded_df = pd.DataFrame(X_demo, columns=[f"feature_{i}" for i in range(X_demo.shape[1])])
    loaded_df["target"] = y_demo
    return loaded_df


df = load_data(config)
print(df.shape)
df.head()

## 7. Exploratory Data Analysis

**Why:** you need to know two things before writing a single line of model
code — is anything missing, and (for classification) is the target
imbalanced? Both change decisions later (imputation strategy, whether to pass
`class_weights` into the loss function).

**Common mistake:** looking only at overall accuracy on an imbalanced dataset.
A dataset that's 95% one class gets 95% "accuracy" by always predicting that
class — this is exactly why section 19 computes precision/recall/F1 too, not
just accuracy.


In [ ]:
print(df.info())
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")

if config.task_type == "classification":
    ax = df["target"].value_counts().sort_index().plot(kind="bar", figsize=(6, 4))
    ax.set_title("Class distribution")
    ax.set_xlabel("Class")
    ax.set_ylabel("Count")
else:
    ax = df["target"].plot(kind="hist", bins=40, figsize=(6, 4))
    ax.set_title("Target distribution")
plt.show()

df.describe().T.head(10)

## 8. Data Preprocessing

**Why the scaler is only *defined* here, not fit yet:** `StandardScaler`
learns a mean/std from whatever data you call `.fit()` on. Fitting it on the
full dataset before splitting would let test-set statistics leak into
training — a subtle, very common form of data leakage that inflates your
reported metrics. The actual `.fit_transform()` happens in the next cell,
**after** the split, on the training data only.

**Common mistake:** imputing missing values (mean/median) on the whole
dataset before splitting — same leakage problem as scaling, just less
obvious. In production you'd fit a `SimpleImputer` on train only, exactly
like the scaler below.


In [ ]:
feature_columns = [c for c in df.columns if c != "target"]

if df[feature_columns].isnull().any().any():
    print("Missing values detected - filling with training-set medians "
          "would be the leakage-safe approach; here we fill with the "
          "column median as a simple placeholder.")
    df[feature_columns] = df[feature_columns].fillna(df[feature_columns].median())

X = df[feature_columns].values.astype(np.float32)
y_raw = df["target"].values

if config.task_type == "classification":
    y = y_raw.astype(np.int64)
    num_classes = len(np.unique(y))
else:
    y = y_raw.astype(np.float32).reshape(-1, 1)
    num_classes = 1

print(f"X: {X.shape}   y: {y.shape}")

# Defined now, fit later (after the split) - see markdown above.
scaler = StandardScaler()

## 9. Train / Validation / Test Split

**Why three sets, not two:** the validation set is what Optuna tunes
against in section 23. The test set is never seen until section 25 — exactly
once. If you tune against the test set (a mistake this template deliberately
avoids), your final reported accuracy is optimistic because you've indirectly
fit hyperparameters to it.

**Common mistake:** forgetting `stratify=y` on a classification problem —
without it, a small or imbalanced class can end up disproportionately
represented in one split, especially with a small dataset.

**Best practice:** fit the scaler with `.fit_transform()` on train, then only
ever `.transform()` (never re-fit) on val and test.


In [ ]:
stratify_full = y if config.task_type == "classification" else None

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=(config.test_size + config.val_size),
    random_state=config.seed, stratify=stratify_full,
)

relative_test_size = config.test_size / (config.test_size + config.val_size)
stratify_temp = y_temp if config.task_type == "classification" else None

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=relative_test_size,
    random_state=config.seed, stratify=stratify_temp,
)

# Keep raw (unscaled) copies around - used later in section 27 to demonstrate
# inference on genuinely raw, never-before-scaled data.
X_test_raw = X_test.copy()

X_train = scaler.fit_transform(X_train)   # fit ONLY on train
X_val = scaler.transform(X_val)           # transform, never re-fit
X_test = scaler.transform(X_test)         # transform, never re-fit

print(f"Train: {X_train.shape}   Val: {X_val.shape}   Test: {X_test.shape}")

## 10. Custom Dataset Class

**Why:** wrapping arrays in a `Dataset` is what lets `DataLoader` batch,
shuffle, and (optionally) parallelize loading for you, instead of writing
manual batching loops.

**Common mistake:** using the wrong target dtype. `nn.CrossEntropyLoss`
requires integer class labels (`torch.long`); `nn.MSELoss`/`SmoothL1Loss`
require float targets. Getting this wrong throws a dtype error deep inside
the loss function, far from where the actual mistake was made.


In [ ]:
class TabularDataset(Dataset):
    """Generic Dataset for tabular features + a single target column.

    Works for classification (integer labels) or regression (float targets) -
    the dtype is decided once here based on config.task_type.
    """

    def __init__(self, features: np.ndarray, targets: np.ndarray):
        self.features = torch.as_tensor(features, dtype=torch.float32)
        target_dtype = torch.long if config.task_type == "classification" else torch.float32
        self.targets = torch.as_tensor(targets, dtype=target_dtype)

    def __len__(self) -> int:
        return len(self.features)

    def __getitem__(self, idx: int) -> Tuple[torch.Tensor, torch.Tensor]:
        return self.features[idx], self.targets[idx]


train_dataset = TabularDataset(X_train, y_train)
val_dataset = TabularDataset(X_val, y_val)
test_dataset = TabularDataset(X_test, y_test)
print(f"{len(train_dataset)} train / {len(val_dataset)} val / {len(test_dataset)} test samples")

## 11. DataLoader

**Why these specific arguments:**
- `pin_memory=True` (CUDA only) speeds up host-to-GPU transfer.
- `persistent_workers=True` avoids respawning worker processes every epoch
  when `num_workers > 0`.
- `shuffle=True` only for training — shuffling val/test wastes compute and
  makes predictions harder to line up with their original rows for debugging.

**Common mistake:** setting `num_workers > 0` on Windows or directly in a
notebook without an `if __name__ == "__main__":` guard can hang. If you hit
that, set `config.num_workers = 0`.


In [ ]:
def make_dataloader(dataset: Dataset, batch_size: int, shuffle: bool, config: Config) -> DataLoader:
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        num_workers=config.num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=(config.num_workers > 0),
        drop_last=False,
    )


train_loader = make_dataloader(train_dataset, config.batch_size, shuffle=True, config=config)
val_loader = make_dataloader(val_dataset, config.batch_size, shuffle=False, config=config)
test_loader = make_dataloader(test_dataset, config.batch_size, shuffle=False, config=config)
print(f"{len(train_loader)} train batches / {len(val_loader)} val batches / {len(test_loader)} test batches")

## 12. Model Architecture

**Why `hidden_dims` is a list:** the same class handles a 1-layer network or
a 5-layer one just by changing a config value — no architecture code changes.
This is exactly the shape Optuna needs later to search over depth *and*
width.

**Common mistake:** adding a final `nn.Softmax()`/`nn.Sigmoid()` after the
last layer. `nn.CrossEntropyLoss` expects raw logits and applies
log-softmax internally — softmaxing twice silently cripples training (the
gradients become tiny and the model barely learns).


In [ ]:
class MLP(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, hidden_dims: List[int],
                 dropout_rate: float = 0.2, use_batch_norm: bool = True):
        super().__init__()
        layers = []
        in_dim = input_dim
        for h in hidden_dims:
            layers.append(nn.Linear(in_dim, h))
            if use_batch_norm:
                layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU(inplace=True))
            if dropout_rate > 0:
                layers.append(nn.Dropout(dropout_rate))
            in_dim = h
        layers.append(nn.Linear(in_dim, output_dim))  # raw logits / raw regression output - no activation
        self.net = nn.Sequential(*layers)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


input_dim = X_train.shape[1]
output_dim = num_classes if config.task_type == "classification" else 1
print(f"input_dim={input_dim}  output_dim={output_dim}")

## 13. Weight Initialization

**Why:** PyTorch's default `Linear` init is reasonable, but Kaiming/He
initialization is specifically derived for ReLU-family activations and tends
to give more stable gradients early in training, especially in deeper MLPs.

**Common mistake:** applying a custom init to `BatchNorm` layers too. Leave
BatchNorm's default (weight=1, bias=0) alone — it's already correct and
overriding it can destabilize early training.


In [ ]:
def init_weights(module: nn.Module) -> None:
    if isinstance(module, nn.Linear):
        nn.init.kaiming_normal_(module.weight, nonlinearity="relu")
        if module.bias is not None:
            nn.init.zeros_(module.bias)


demo_model = MLP(input_dim, output_dim, config.hidden_dims, config.dropout_rate, config.use_batch_norm)
demo_model.apply(init_weights)
demo_model

## 14. Loss Function Selection

**Why `label_smoothing` for classification:** a small, cheap regularizer that
discourages over-confident predictions and tends to improve calibration —
close to a default choice in 2026 for classification tasks.

**Why `SmoothL1Loss` (Huber) for regression:** less sensitive to outliers
than plain MSE, since it behaves like MAE for large errors and MSE for small
ones. Swap to `nn.MSELoss()` if you specifically want to penalize large
errors more heavily.

**Common mistake:** mismatching loss and final-layer activation — see the
softmax warning in section 12.


In [ ]:
def get_loss_fn(task_type: str, class_weights: Optional[torch.Tensor] = None) -> nn.Module:
    if task_type == "classification":
        return nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.05)
    elif task_type == "regression":
        return nn.SmoothL1Loss()
    raise ValueError(f"Unknown task_type: {task_type}")


criterion = get_loss_fn(config.task_type)
criterion

## 15. Optimizer Selection

**Why `AdamW` is the default:** it decouples weight decay from the gradient
update (in plain `Adam`, "weight_decay" is really L2 regularization folded
into the gradient, which interacts poorly with adaptive learning rates).
AdamW is the standard default for most ANN/tabular workloads in 2026.

**The single most common bug this pattern prevents:** constructing an
optimizer variant inside an `if`/`elif` block and forgetting to reassign it
back to the variable actually used in the training loop —

```python
optimizer = optim.SGD(model.parameters(), lr=0.1)   # placeholder
if optimizer_name == "Adam":
    optim.Adam(model.parameters(), lr=lr)           # BUG: never assigned!
```

Every trial then silently trains with the placeholder SGD regardless of what
a hyperparameter search suggests. Writing this as a function that *returns* a
value, always captured with `optimizer = get_optimizer(...)`, makes that bug
structurally impossible.


In [ ]:
def get_optimizer(name: str, params, lr: float, weight_decay: float) -> optim.Optimizer:
    name = name.lower()
    if name == "adamw":
        return optim.AdamW(params, lr=lr, weight_decay=weight_decay)
    elif name == "adam":
        return optim.Adam(params, lr=lr, weight_decay=weight_decay)
    elif name == "sgd":
        return optim.SGD(params, lr=lr, weight_decay=weight_decay, momentum=0.9, nesterov=True)
    elif name == "rmsprop":
        return optim.RMSprop(params, lr=lr, weight_decay=weight_decay)
    raise ValueError(f"Unknown optimizer: {name}")

## 16. Learning Rate Scheduler

**Why this returns a `step_unit` alongside the scheduler:** not all
schedulers step at the same granularity, and stepping at the wrong one is a
common, silent bug:
- `OneCycleLR` must be stepped every **batch**, not every epoch.
- `ReduceLROnPlateau` must be stepped every **epoch**, and needs a metric
  value passed in.
- `StepLR`/`CosineAnnealingLR` step once per **epoch**, no metric needed.

Returning `("batch" | "epoch" | "epoch_metric")` lets the training loop
(section 17/21) call `.step()` correctly without guessing.


In [ ]:
def get_scheduler(name: str, optimizer: optim.Optimizer, config: Config,
                   steps_per_epoch: Optional[int] = None):
    name = name.lower()
    if name == "none":
        return None, None
    if name == "step":
        return optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5), "epoch"
    if name == "cosine":
        return optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=config.max_epochs), "epoch"
    if name == "plateau":
        return optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", factor=0.5, patience=3), "epoch_metric"
    if name == "onecycle":
        if steps_per_epoch is None:
            raise ValueError("The 'onecycle' scheduler needs steps_per_epoch.")
        sched = optim.lr_scheduler.OneCycleLR(
            optimizer, max_lr=config.learning_rate,
            steps_per_epoch=steps_per_epoch, epochs=config.max_epochs,
        )
        return sched, "batch"
    raise ValueError(f"Unknown scheduler: {name}")

## 17. Training Loop

**Why `zero_grad(set_to_none=True)`:** setting gradients to `None` instead of
zero-filling a tensor avoids an unnecessary memory write every step — a small
but free speedup.

**Why the AMP plumbing lives here already** (rather than bolted on later in
section 22): retrofitting mixed precision into an existing loop is exactly
how AMP bugs happen (forgetting `scaler.scale()`/`scaler.update()`, or
autocast wrapping the loss computation but not the forward pass). Building it
in from the start, gated by a boolean, means section 22 is just "flip
`config.use_amp = True`" rather than a rewrite.

**Common mistake:** calling a batch-stepped scheduler (like `OneCycleLR`)
once per epoch instead of once per batch — it silently finishes its entire
cycle in the first epoch or two.


In [ ]:
def train_one_epoch(model, loader, optimizer, criterion, device,
                     scheduler=None, step_unit: Optional[str] = None,
                     scaler: Optional[torch.amp.GradScaler] = None) -> float:
    model.train()
    running_loss = 0.0
    use_amp = scaler is not None

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with torch.amp.autocast(device_type=device.type, enabled=use_amp):
            outputs = model(xb)
            loss = criterion(outputs, yb)

        if use_amp:
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            optimizer.step()

        if step_unit == "batch" and scheduler is not None:
            scheduler.step()

        running_loss += loss.item() * xb.size(0)

    return running_loss / len(loader.dataset)

## 18. Validation Loop

**Why `@torch.no_grad()` and `model.eval()` are both needed, not just one:**
`model.eval()` switches BatchNorm/Dropout to inference behavior;
`torch.no_grad()` stops PyTorch from building a backward graph. Forgetting
`eval()` silently uses training-mode BatchNorm statistics on val/test data;
forgetting `no_grad()` wastes memory (and, on GPU, some time) building a
graph that's never used.


In [ ]:
@torch.no_grad()
def evaluate(model, loader, criterion, device, task_type: str):
    model.eval()
    running_loss = 0.0
    all_preds, all_targets = [], []

    for xb, yb in loader:
        xb = xb.to(device, non_blocking=True)
        yb = yb.to(device, non_blocking=True)

        outputs = model(xb)
        loss = criterion(outputs, yb)
        running_loss += loss.item() * xb.size(0)

        preds = outputs.argmax(dim=1) if task_type == "classification" else outputs.squeeze(-1)
        all_preds.append(preds.cpu())
        all_targets.append(yb.cpu())

    avg_loss = running_loss / len(loader.dataset)
    return avg_loss, torch.cat(all_preds).numpy(), torch.cat(all_targets).numpy()

## 19. Metrics

**Why weighted precision/recall/F1, not just accuracy:** as flagged in the
EDA section, accuracy alone is misleading on imbalanced data. `average="weighted"`
accounts for class support so metrics reflect the actual class distribution.

**Why R2 for regression:** MSE/MAE are in the target's raw units and hard to
judge in isolation (is an MAE of 12 good or bad?). R2 gives a
scale-independent sense of "how much better than predicting the mean" the
model is doing.


In [ ]:
def compute_metrics(y_true: np.ndarray, y_pred: np.ndarray, task_type: str) -> Dict[str, float]:
    if task_type == "classification":
        precision, recall, f1, _ = precision_recall_fscore_support(
            y_true, y_pred, average="weighted", zero_division=0
        )
        return {"accuracy": accuracy_score(y_true, y_pred), "precision": precision,
                "recall": recall, "f1": f1}
    return {"mse": mean_squared_error(y_true, y_pred), "mae": mean_absolute_error(y_true, y_pred),
            "r2": r2_score(y_true, y_pred)}

## 20. Early Stopping

**Why it keeps a copy of the best weights, not just the best score:** the
whole point of early stopping is to roll back to the epoch that generalized
best. Tracking only the *number* and stopping when it stalls, without also
restoring those weights, leaves the model at its LAST (possibly already
overfit) epoch instead of its best one — a surprisingly common version of
this bug.

**Best practice / scaling note:** `copy.deepcopy(model.state_dict())` keeps
the best weights in CPU/GPU memory, which is fine for small-to-medium ANNs.
For very large models, checkpoint the best weights to disk instead, to avoid
holding two full copies in memory at once.


In [ ]:
class EarlyStopping:
    def __init__(self, patience: int = 7, mode: str = "min", min_delta: float = 1e-4):
        self.patience = patience
        self.mode = mode
        self.min_delta = min_delta
        self.counter = 0
        self.best_score: Optional[float] = None
        self.best_epoch: int = 0
        self.best_state: Optional[dict] = None
        self.should_stop = False

    def step(self, value: float, epoch: int, model: nn.Module) -> bool:
        improved = (
            self.best_score is None
            or (self.mode == "min" and value < self.best_score - self.min_delta)
            or (self.mode == "max" and value > self.best_score + self.min_delta)
        )
        if improved:
            self.best_score = value
            self.best_epoch = epoch
            self.best_state = copy.deepcopy(model.state_dict())
            self.counter = 0
        else:
            self.counter += 1
            self.should_stop = self.counter >= self.patience
        return self.should_stop

## 21. Model Checkpointing

**Why save `state_dict()`, not the whole model object:** pickling the entire
`nn.Module` ties the checkpoint to the exact class definition and file layout
at save time — rename a class or move a file later and old checkpoints can
fail to load. Saving just the weights (plus enough metadata to reconstruct
the architecture) is the portable, industry-standard approach.


In [ ]:
def save_checkpoint(path: str, model: nn.Module, extra: Optional[dict] = None) -> None:
    payload = {"model_state_dict": model.state_dict()}
    if extra:
        payload.update(extra)
    torch.save(payload, path)


def load_checkpoint(path: str, map_location=None) -> dict:
    return torch.load(path, map_location=map_location)

### Putting sections 14–21 together: `fit_model`

One function used for the baseline run below, every Optuna trial (section
23), and the final retrain (section 24) — instead of three copies of the
same loop slowly drifting apart.

- `val_loader=None` → no early stopping, no pruning, trains for exactly
  `max_epochs`. This is what the final retrain (section 24) uses, since by
  then there's no held-out validation data left.
- `trial` passed → reports validation loss every epoch so Optuna's pruner can
  cut a clearly unpromising trial short.
- Validation loss (not accuracy) is what the scheduler, early stopping, and
  Optuna all monitor — loss is always "lower is better" for both classification
  and regression, which avoids flipping comparison direction between tasks.


In [ ]:
def fit_model(model: nn.Module, train_loader: DataLoader, criterion: nn.Module, config: Config,
              optimizer_name: str, learning_rate: float, weight_decay: float,
              val_loader: Optional[DataLoader] = None, scheduler_name: str = "none",
              max_epochs: int = 50, patience: int = 7, use_amp: bool = False,
              trial: Optional[optuna.Trial] = None, verbose: bool = False) -> Dict[str, Any]:
    if scheduler_name == "plateau" and val_loader is None:
        raise ValueError("The 'plateau' scheduler needs a validation metric - pass a val_loader.")

    optimizer = get_optimizer(optimizer_name, model.parameters(), learning_rate, weight_decay)
    scheduler, step_unit = get_scheduler(scheduler_name, optimizer, config, steps_per_epoch=len(train_loader))
    scaler = torch.amp.GradScaler(device.type) if (use_amp and device.type == "cuda") else None

    stopper = EarlyStopping(patience=patience, mode="min") if val_loader is not None else None
    history: Dict[str, list] = {"train_loss": [], "val_loss": []}

    for epoch in range(max_epochs):
        train_loss = train_one_epoch(
            model, train_loader, optimizer, criterion, device,
            scheduler=scheduler if step_unit == "batch" else None,
            step_unit=step_unit, scaler=scaler,
        )
        history["train_loss"].append(train_loss)

        if val_loader is not None:
            val_loss, _, _ = evaluate(model, val_loader, criterion, device, config.task_type)
            history["val_loss"].append(val_loss)

            if step_unit == "epoch":
                scheduler.step()
            elif step_unit == "epoch_metric":
                scheduler.step(val_loss)

            if trial is not None:
                trial.report(val_loss, epoch)
                if trial.should_prune():
                    raise optuna.TrialPruned()

            if verbose:
                print(f"Epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")

            if stopper.step(val_loss, epoch, model):
                if verbose:
                    print(f"Early stopping at epoch {epoch+1} (best was epoch {stopper.best_epoch+1})")
                break
        else:
            if step_unit == "epoch":
                scheduler.step()
            if verbose and (epoch + 1) % 10 == 0:
                print(f"Epoch {epoch+1}/{max_epochs}  train_loss={train_loss:.4f}")

    best_epoch = stopper.best_epoch if stopper is not None else max_epochs - 1
    if stopper is not None and stopper.best_state is not None:
        model.load_state_dict(stopper.best_state)  # roll back to the BEST epoch, not the last one

    return {"model": model, "history": history, "best_epoch": best_epoch}

### Baseline sanity-check run

Trains once with the untouched `Config` defaults, before any hyperparameter
search — confirms the whole pipeline (sections 12–21) actually works, and
gives a reference point to compare the tuned model against later.


In [ ]:
baseline_model = MLP(input_dim, output_dim, config.hidden_dims, config.dropout_rate, config.use_batch_norm).to(device)
baseline_model.apply(init_weights)

baseline_result = fit_model(
    baseline_model, train_loader, criterion, config,
    optimizer_name=config.optimizer_name, learning_rate=config.learning_rate,
    weight_decay=config.weight_decay, val_loader=val_loader, scheduler_name=config.scheduler_name,
    max_epochs=config.max_epochs, patience=config.early_stopping_patience,
    use_amp=config.use_amp, trial=None, verbose=True,
)

plt.figure(figsize=(6, 4))
plt.plot(baseline_result["history"]["train_loss"], label="train")
plt.plot(baseline_result["history"]["val_loss"], label="val")
plt.xlabel("epoch"); plt.ylabel("loss"); plt.legend(); plt.title("Baseline training curves")
plt.show()

_, baseline_preds, baseline_targets = evaluate(baseline_model, val_loader, criterion, device, config.task_type)
print("Baseline val metrics:", compute_metrics(baseline_targets, baseline_preds, config.task_type))

## 22. Mixed Precision Training (`torch.amp`)

**What it is:** runs most ops in float16 (or bfloat16) instead of float32,
which roughly halves memory use and can meaningfully speed up training on
modern CUDA GPUs (Tensor Cores), at essentially no accuracy cost thanks to
`GradScaler` protecting against float16 underflow in the backward pass.

**Why there's no new training loop here:** `fit_model`/`train_one_epoch`
already accept a `use_amp` flag (section 17) — enabling it is just
`config.use_amp = True`, not a rewrite.

**Common mistake:** enabling AMP on CPU. `autocast` technically works there
(bfloat16), but the real speedup is GPU Tensor Cores — on CPU it's a no-op
at best, so we only actually engage `GradScaler` when `device.type == "cuda"`.


In [ ]:
if device.type == "cuda":
    print("CUDA detected - set config.use_amp = True to enable mixed precision in fit_model.")
else:
    print(f"Running on {device.type} - AMP mainly benefits CUDA GPUs, "
          "so config.use_amp would be a no-op here.")

## 23. Hyperparameter Tuning with Optuna

**Fixes applied vs. a naive implementation:**
- **Proper validation set** — `objective` only ever touches `train_dataset`
  and `val_dataset`. `test_dataset` doesn't appear anywhere in this section.
- **Pruning** — `MedianPruner` stops a trial early if it's clearly
  underperforming others at the same epoch, so the same trial budget covers
  more of the search space instead of running bad configs to completion.
- **Correct optimizer implementation** — `fit_model` calls
  `get_optimizer(...)` and uses the returned object directly; there's no
  hardcoded optimizer sitting in scope to silently fall back to.

`config.n_optuna_trials` defaults to a modest 15 to keep this template fast to
run end to end — for a real project, treat that as a starting point and
increase it (along with search-space width) once you know the pipeline works.


In [ ]:
def objective(trial: optuna.Trial) -> float:
    n_layers = trial.suggest_int("n_layers", 1, 4)
    hidden_dims = [trial.suggest_int(f"n_units_l{i}", 16, 256, step=16) for i in range(n_layers)]
    dropout_rate = trial.suggest_float("dropout_rate", 0.0, 0.5, step=0.1)
    optimizer_name = trial.suggest_categorical("optimizer_name", ["adamw", "sgd", "rmsprop"])
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)
    weight_decay = trial.suggest_float("weight_decay", 1e-6, 1e-2, log=True)
    batch_size = trial.suggest_categorical("batch_size", [32, 64, 128])
    scheduler_name = trial.suggest_categorical("scheduler_name", ["cosine", "plateau", "none"])

    trial_train_loader = make_dataloader(train_dataset, batch_size, shuffle=True, config=config)
    trial_val_loader = make_dataloader(val_dataset, batch_size, shuffle=False, config=config)

    trial_model = MLP(input_dim, output_dim, hidden_dims, dropout_rate, config.use_batch_norm).to(device)
    trial_model.apply(init_weights)
    trial_criterion = get_loss_fn(config.task_type)

    result = fit_model(
        trial_model, trial_train_loader, trial_criterion, config,
        optimizer_name=optimizer_name, learning_rate=learning_rate, weight_decay=weight_decay,
        val_loader=trial_val_loader, scheduler_name=scheduler_name,
        max_epochs=config.max_epochs, patience=config.early_stopping_patience,
        use_amp=config.use_amp, trial=trial, verbose=False,
    )

    trial.set_user_attr("best_epoch", result["best_epoch"])
    final_val_loss, _, _ = evaluate(result["model"], trial_val_loader, trial_criterion, device, config.task_type)
    return final_val_loss

In [ ]:
pruner = optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=3)
sampler = optuna.samplers.TPESampler(seed=config.seed)

study = optuna.create_study(direction="minimize", sampler=sampler, pruner=pruner)
study.optimize(objective, n_trials=config.n_optuna_trials, timeout=config.optuna_timeout_sec)

n_pruned = len(study.get_trials(states=[TrialState.PRUNED]))
n_complete = len(study.get_trials(states=[TrialState.COMPLETE]))
print(f"Completed: {n_complete}  Pruned: {n_pruned}")
print(f"Best val loss: {study.best_value:.4f}")
print(f"Best params: {study.best_params}")

In [ ]:
try:
    from optuna.visualization.matplotlib import plot_optimization_history, plot_param_importances
    plot_optimization_history(study); plt.show()
    plot_param_importances(study); plt.show()
except ImportError:
    print("Install plotly for Optuna's visualization module: pip install plotly")

## 24. Retrain Best Model

**Why combine train + val here:** hyperparameters (including how many epochs
to run, taken from the best trial's early-stopping point) are now fixed, so
there's no more need to hold back validation data — using every non-test
sample gives the final model strictly more data to learn from.

**Why `plateau` gets swapped for `cosine` if it was the winning scheduler:**
`ReduceLROnPlateau` needs a validation metric to react to, and there's no
held-out validation set left at this stage — falling back to a schedule that
doesn't need one keeps this step from breaking on an otherwise-fine set of
hyperparameters.


In [ ]:
best_params = study.best_params
best_hidden_dims = [best_params[f"n_units_l{i}"] for i in range(best_params["n_layers"])]
best_epoch_count = study.best_trial.user_attrs["best_epoch"] + 1  # +1: epochs are 0-indexed internally

final_scheduler_name = best_params["scheduler_name"]
if final_scheduler_name == "plateau":
    final_scheduler_name = "cosine"  # no val metric left to react to

X_train_full = np.concatenate([X_train, X_val])
y_train_full = np.concatenate([y_train, y_val])
train_full_dataset = TabularDataset(X_train_full, y_train_full)
train_full_loader = make_dataloader(train_full_dataset, best_params["batch_size"], shuffle=True, config=config)

final_model = MLP(input_dim, output_dim, best_hidden_dims, best_params["dropout_rate"], config.use_batch_norm).to(device)
final_model.apply(init_weights)
final_criterion = get_loss_fn(config.task_type)

final_result = fit_model(
    final_model, train_full_loader, final_criterion, config,
    optimizer_name=best_params["optimizer_name"], learning_rate=best_params["learning_rate"],
    weight_decay=best_params["weight_decay"], val_loader=None, scheduler_name=final_scheduler_name,
    max_epochs=best_epoch_count, patience=config.early_stopping_patience,
    use_amp=config.use_amp, trial=None, verbose=True,
)
final_model = final_result["model"]

## 25. Final Evaluation on Test Set

This is the **only** cell in the whole notebook that touches `test_loader`.
That's deliberate — it's what makes this number an honest estimate of
real-world performance rather than one that's been indirectly tuned to.


In [ ]:
test_loader_best = make_dataloader(test_dataset, best_params["batch_size"], shuffle=False, config=config)
test_loss, test_preds, test_targets = evaluate(final_model, test_loader_best, final_criterion, device, config.task_type)

metrics = compute_metrics(test_targets, test_preds, config.task_type)
print(f"Test loss: {test_loss:.4f}")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

if config.task_type == "classification":
    print("\n" + classification_report(test_targets, test_preds, zero_division=0))
    cm = confusion_matrix(test_targets, test_preds)
    fig, ax = plt.subplots(figsize=(6, 6))
    ConfusionMatrixDisplay(confusion_matrix=cm).plot(ax=ax, cmap="Blues", colorbar=False)
    plt.title("Test set confusion matrix")
    plt.show()
else:
    plt.figure(figsize=(6, 6))
    plt.scatter(test_targets, test_preds, alpha=0.4)
    lims = [min(test_targets.min(), test_preds.min()), max(test_targets.max(), test_preds.max())]
    plt.plot(lims, lims, "r--")
    plt.xlabel("True value"); plt.ylabel("Predicted value")
    plt.title("Test set: predicted vs. true")
    plt.show()

## 26. Save Best Model

**Why the scaler's mean/scale are saved alongside the weights:** a checkpoint
is only useful if you can reproduce the *exact* preprocessing at inference
time. Saving the model without the scaler parameters is a common mistake
that leads to silently wrong predictions later, once nobody remembers which
scaling was used.


In [ ]:
checkpoint_path = os.path.join(config.output_dir, config.checkpoint_name)

save_checkpoint(
    checkpoint_path, final_model,
    extra={
        "input_dim": input_dim,
        "output_dim": output_dim,
        "hidden_dims": best_hidden_dims,
        "dropout_rate": best_params["dropout_rate"],
        "use_batch_norm": config.use_batch_norm,
        "task_type": config.task_type,
        "scaler_mean": scaler.mean_.tolist(),
        "scaler_scale": scaler.scale_.tolist(),
        "feature_columns": feature_columns,
        "best_params": best_params,
        "test_metrics": metrics,
    },
)
print(f"Saved model + preprocessing info to {checkpoint_path}")

## 27. Inference on New Data

**Why this loads the scaler's mean/scale from the checkpoint instead of
reusing the in-memory `scaler` object:** it proves the checkpoint is actually
self-contained — this is exactly the code path a separate inference script
(with no access to this notebook's variables) would run.

**Common mistake this avoids:** calling `scaler.fit_transform()` on new data
at inference time. You must reuse the *exact* mean/scale learned on the
training data — fitting a fresh scaler on a single incoming row (or a
differently-distributed batch) silently produces wrong predictions.


In [ ]:
def predict(raw_features: np.ndarray, checkpoint_path: str) -> np.ndarray:
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)

    mean = np.array(checkpoint["scaler_mean"], dtype=np.float32)
    scale = np.array(checkpoint["scaler_scale"], dtype=np.float32)
    scaled = (raw_features.astype(np.float32) - mean) / scale

    inference_model = MLP(
        input_dim=checkpoint["input_dim"], output_dim=checkpoint["output_dim"],
        hidden_dims=checkpoint["hidden_dims"], dropout_rate=checkpoint["dropout_rate"],
        use_batch_norm=checkpoint["use_batch_norm"],
    ).to(device)
    inference_model.load_state_dict(checkpoint["model_state_dict"])
    inference_model.eval()

    with torch.no_grad():
        x = torch.as_tensor(scaled, dtype=torch.float32).to(device)
        outputs = inference_model(x)
        if checkpoint["task_type"] == "classification":
            return outputs.argmax(dim=1).cpu().numpy()
        return outputs.squeeze(-1).cpu().numpy()


# Demo: treat the first 5 raw (never-scaled) test rows as brand-new incoming data
sample_raw = X_test_raw[:5]
sample_true = y_test[:5].ravel()
sample_preds = predict(sample_raw, checkpoint_path)

print("True:     ", sample_true)
print("Predicted:", sample_preds)

In [ ]:
print("=" * 50)
print("SUMMARY")
print("=" * 50)
print(f"Task type:            {config.task_type}")
print(f"Optuna trials run:     {n_complete} completed, {n_pruned} pruned")
print(f"Best hyperparameters:  {best_params}")
print(f"Final test metrics:    {metrics}")
print(f"Model checkpoint:      {checkpoint_path}")

## 28. Summary & Next Steps

**To reuse this template on a new problem:**
1. Set `config.task_type` and `config.data_path` (section 3).
2. If your CSV's target column isn't named `target`, set `config.target_column`.
3. Restart & Run All.

**Things worth adding as a project grows beyond this template:**
- **Experiment tracking** (MLflow / Weights & Biases) once you're running
  enough experiments that `study.best_params` in a notebook cell stops being
  enough to track what happened.
- **Cross-validation** instead of a single train/val split, if the dataset is
  small enough that one split's luck meaningfully affects the result.
- **A different architecture entirely** for non-tabular data — this template
  is an MLP over flat feature vectors; images generally want a CNN, sequences
  generally want a Transformer/RNN, and an MLP on flattened pixels is a
  reasonable learning exercise but not how production image models are built.
- **Pipeline orchestration** (a script + config file, or tools like Hydra/
  ZenML) once this needs to run unattended rather than cell-by-cell.
